# Complete Hybrid MPI-CUDA SBFC application
This notebook clones the GitHub project, builds the CUDA/MPI engine and React interface, starts the API, and embeds the complete application in Colab. Before running it, choose **Runtime > Change runtime type > T4 GPU**. The application remains available while this runtime is connected.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPOSITORY.git'
REPO_BRANCH = 'main'
PROJECT_DIR = Path('/content/sbfc-app')

if 'YOUR_USERNAME' in REPO_URL:
    raise ValueError('Replace REPO_URL with your GitHub repository URL, then run this cell again.')
if PROJECT_DIR.exists():
    raise RuntimeError(f'{PROJECT_DIR} already exists. Restart the runtime for a clean deployment.')
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
print('Project cloned into', PROJECT_DIR)

In [ ]:
import shutil
import subprocess

subprocess.run(['nvidia-smi'], check=True)
if shutil.which('nvcc') is None:
    raise RuntimeError('CUDA compiler not found. Select a GPU runtime and reconnect.')
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'openmpi-bin', 'libopenmpi-dev', 'nodejs', 'npm'], check=True)
print('CUDA, MPI, Node.js, and npm are ready')

In [ ]:
import shlex
import subprocess

engine_dir = PROJECT_DIR / 'engine' / 'bin'
engine_dir.mkdir(parents=True, exist_ok=True)
mpi_cflags = shlex.split(subprocess.check_output(['mpicxx', '--showme:compile'], text=True))
mpi_ldflags = shlex.split(subprocess.check_output(['mpicxx', '--showme:link'], text=True))
compile_command = [
    'nvcc', '-O3', '-std=c++14', *mpi_cflags,
    str(PROJECT_DIR / 'engine' / 'src' / 'sbfc_hybrid.cu'),
    '-o', str(engine_dir / 'sbfc-hybrid'), *mpi_ldflags,
]
print('Compiling hybrid engine...')
subprocess.run(compile_command, check=True)
print('Building React interface...')
subprocess.run(['npm', 'ci'], cwd=PROJECT_DIR / 'frontend', check=True)
subprocess.run(['npm', 'run', 'build'], cwd=PROJECT_DIR / 'frontend', check=True)
print('Application build completed')

In [ ]:
import os
import subprocess
import time
import urllib.request

log_path = Path('/content/sbfc-server.log')
server_log = log_path.open('w')
server_env = os.environ.copy()
server_env['SBFC_HOST'] = '0.0.0.0'
server_env['SBFC_PORT'] = '8000'
app_server = subprocess.Popen(
    ['python3', 'server/server.py'], cwd=PROJECT_DIR, env=server_env,
    stdout=server_log, stderr=subprocess.STDOUT,
)
for _ in range(60):
    if app_server.poll() is not None:
        server_log.flush()
        raise RuntimeError(log_path.read_text())
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/', timeout=1)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Application did not start. Inspect /content/sbfc-server.log')
print('Application server is running')

## Open the application
Keep the runtime connected and this notebook open. On a standard Colab T4 runtime, leave **MPI ranks** set to `1`. The graph preview is bounded, but the engine simulates the full node count.

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8000, width='100%', height=950)

In [ ]:
# Run this only when debugging a failed request.
print(Path('/content/sbfc-server.log').read_text()[-5000:])